# CKA-RL on Meta-World — Kaggle runner**Panel settings:** Accelerator = **GPU T4 x2**, Internet = **On**, then`Save Version` → **Save & Run All (Commit)** and close the tab. Kaggle runs it inthe background; limits are **12 h per GPU session** and **20 GB** of auto-saved`/kaggle/working`.**Run ONE stage per commit.** Every idea is its own stage, so nothing has to fitthe 12 h wall and stages split freely across two accounts.| order | STAGE | who | roughly | what it answers ||---|---|---|---|---|| 1 | `smoke` | A | ~10 min | does the pipeline run at all || 2 | `pilot` | A | ~1 h | do these tasks learn in 150k steps || 3 | `baselines s0` | both | ~1.5 h | FT denominators || 4 | `cond s0 1` / `cond s0 2` | A | bulk | classic_cka, ± distillation || 4 | `cond s0 3` / `cond s0 4` | B | bulk | weight_delta, ± distillation || 5 | `pretrain` → `baselines s4` → `cond s4 N` | day 2 | optional | shared-layer idea || 6 | `report s0` | one account | minutes | after merging outputs |**Do not skip `smoke` and `pilot`.** Smoke catches wiring bugs in ten minutesinstead of after an hour. Pilot decides whether the 150k budget is viable at all —if the tasks do not learn, every later comparison is comparing noise.

## 1. Config — the only cell you normally edit

In [ ]:
REPO_URL    = "https://github.com/Yasamin-Rajabi/Continual-RL-Project.git"REPO_BRANCH = "Narges"REPO_SUBDIR = "metaworld"     # folder inside the repo; "" if the code is at the rootSTAGE = "smoke"               # smoke | pilot | baselines | cond | report | pretrain | setup | sanityARG1  = "s0"                  # for baselines/cond/report: s0 or s4ARG2  = ""                    # for cond: "1".."4"# Stop cleanly before Kaggle's 12 h wall so the version still COMMITS and the# output is saved. A version killed by the wall is marked failed.BUDGET_HOURS = 10.5import os, pathlibWORK = pathlib.Path("/kaggle/working")CODE = pathlib.Path("/kaggle/temp/repo")   # scratch: not part of the saved outputprint("stage:", STAGE, ARG1, ARG2, "| budget:", BUDGET_HOURS, "h")

## 2. Pull the code (public repo)Cloned into `/kaggle/temp` so every session gets fresh code and the 20 GB outputquota is spent only on results.

In [ ]:
import subprocess, shutilif CODE.exists():    shutil.rmtree(CODE)CODE.parent.mkdir(parents=True, exist_ok=True)subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(CODE)], check=True)PROJ = CODE / REPO_SUBDIR if REPO_SUBDIR else CODEassert (PROJ / "run_kaggle.sh").exists(), f"run_kaggle.sh not in {PROJ} - check REPO_SUBDIR"os.chdir(PROJ)print("code at", PROJ)print(subprocess.run(["git", "log", "-1", "--oneline"], cwd=CODE,                     capture_output=True, text=True).stdout.strip())

## 3. Resume from the previous session`Save & Run All` starts from an empty `/kaggle/working`. To continue a run:`Add-ons → Add data → Your Work / Notebook Output` → pick this notebook's lastversion. It mounts read-only under `/kaggle/input/`; this cell copies it back sothe run manifests are found and finished work is skipped.No-op on the first run.

In [ ]:
import globrestored = 0for src_root in sorted(glob.glob("/kaggle/input/*")):    for name in ("runs", "agents", "analysis", "analysis_scratch",                 "scratch_models", "pretrained_encoders", "plots", "logs"):        src = pathlib.Path(src_root) / name        if not src.is_dir():            continue        shutil.copytree(src, WORK / name, dirs_exist_ok=True)        n = sum(1 for _ in src.rglob("*"))        restored += n        print(f"restored {name} from {src_root} ({n} entries)")print("nothing to restore - first run" if restored == 0 else f"restored {restored} entries")

## 4. InstallMeta-World is installed at a **pinned commit** with `--no-deps`, so pip neverresolves its stale dependency pins and never downgrades torch / numpy /gymnasium underneath the run. MuJoCo goes in first; everything else comes from`requirements.txt` with the `metaworld`/`mujoco` lines filtered out.`MUJOCO_GL=egl` avoids the OpenGL context error some images raise on envconstruction even when nothing is rendered.

In [ ]:
os.environ["MUJOCO_GL"] = "egl"          # try "osmesa" if EGL errors appearos.environ["PYOPENGL_PLATFORM"] = "egl"os.environ["KAGGLE_WORKING"] = str(WORK)!bash run_kaggle.sh setup

## 5. Sanity check (no GPU time)Builds every task and asserts constant obs/action shapes plus the`success` / `task_error` info keys. If this fails, stop — nothing downstreamwill be meaningful.

In [ ]:
!bash run_kaggle.sh sanity

## 6. Run the stage`timeout` stops the work before the wall so the version still commits.**Exit code 124 means "budget reached", not an error** — the next sessionresumes from the manifests.

In [ ]:
import timesecs = int(BUDGET_HOURS * 3600)cmd = f"timeout --signal=INT {secs} bash run_kaggle.sh {STAGE} {ARG1} {ARG2}".strip()print(cmd, flush=True)t0 = time.time()rc = subprocess.run(cmd, shell=True).returncodeelapsed = (time.time() - t0) / 3600print(f"\nstage={STAGE} {ARG1} {ARG2} rc={rc} elapsed={elapsed:.2f} h")if rc == 124:    print("BUDGET REACHED - not a failure. Finished work is checkpointed. "          "Commit this version, add its output as input, run the same stage again.")elif rc != 0:    raise SystemExit(f"stage failed with code {rc} - see the log above")else:    print("stage complete.")

## 7. What was produced

In [ ]:
print(subprocess.run(f"du -sh {WORK}/* 2>/dev/null | sort -h", shell=True,                     capture_output=True, text=True).stdout)print(subprocess.run(f"du -sh {WORK}", shell=True, capture_output=True, text=True).stdout)print("\nNext: Save Version -> Save & Run All (Commit), then add THIS version's "      "output as an input dataset before the next stage.")